# Demo of sampling images using BUSGen

Run the code cell by `Shift+Enter`.

If you are not familiar with Python code, I highly recommend you to try demo at [https://aibus.bio](https://aibus.bio).

In [ ]:
import os
import shutil
import copy
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
from typing import Dict
from PIL import Image
from glob import glob

import torch
import torch.distributed as dist
from torch.cuda.amp import autocast
from torch.nn.parallel import DistributedDataParallel as DDP
from torchvision import transforms
from torchvision.utils import save_image

from BUSGen.DPMSolver import NoiseScheduleVP, DPM_Solver, model_wrapper
from BUSGen.Diffusion import GaussianDiffusionTrainer
from BUSGen.Model import UNet
from BUSGen.BoxSampler import BoxSampler

from Utils import DEVICE_dict_all

class BUSGen:
    def __init__(self, Config: Dict):
        self.Config = Config
        self.device = torch.device("cuda:0")
        os.makedirs(Config["sampled_dir"], exist_ok=True)
        self.model = UNet(
            T=Config["T"], 
            ch=Config["channel"], 
            ch_mult=Config["channel_mult"],
            num_res_blocks=Config["num_res_blocks"],
            dropout=Config["dropout"],
            num_groups=Config["num_groups"], 
            affine=Config["affine"],
            box_cond=Config["box_cond"], 
            cls_cond=Config["cls_cond"], 
            dev_cond=Config["dev_cond"], 
            dev_num=len(DEVICE_dict_all),
        ).cuda()
        
        # Load checkpoint
        ckpt = torch.load(os.path.join(Config["save_weight_dir"], Config["test_load_weight"]), map_location=self.device)
        self.model.load_state_dict(ckpt.get("model", ckpt), strict=True)
        # compile the model
        self.model = torch.compile(self.model)
        self.model.eval()

        self.noise_schedule = NoiseScheduleVP(
            schedule="discrete",
            betas=torch.linspace(Config["beta_1"], Config["beta_T"], Config["T"]).double().to(self.device),
        )
    
    @torch.no_grad()
    def __call__(self, cls_name, dev_name, lesion_box):
        """
        cls_name: 0 for benign, 1 for malignancy
        dev_value: 0 - 17 for different devices
        lesion_box: [x1, y1, x2, y2] in [0, 1]
        """
        cls_value = {"benign": 0, "malignant": 1}[cls_name]
        dev_value = DEVICE_dict_all[dev_name]
        dev_cond = torch.Tensor([dev_value]).long().to(self.device).reshape(1).repeat(self.Config["batch_size"])
        cls_cond = torch.Tensor([cls_value]).long().to(self.device).reshape(1).repeat(self.Config["batch_size"])
        cls_cond = cls_cond + 1
        lesion_box = torch.Tensor(lesion_box).reshape(self.Config["batch_size"], 1, 4).to(self.device)
        lesion_box = lesion_box.clamp(0, 1)
        lesion_box = torch.cat([lesion_box, torch.ones_like(lesion_box[:, :, [0]])], dim=-1)
        noisyImage = torch.randn(
            size=[self.Config["batch_size"], 3, self.Config["img_size"], self.Config["img_size"],], device=self.device
        )
        cond = (lesion_box, cls_cond, dev_cond)
        box, cls, dev = cond
        uncond_box = torch.zeros_like(box).to(self.device)
        uncond_cls = torch.zeros_like(cls).to(self.device).long()
        uncond_dev = torch.zeros_like(dev).to(self.device).long()
        uncond = (uncond_box, uncond_cls, uncond_dev)

        model_fn = model_wrapper(
            self.model,
            self.noise_schedule,
            guidance_type="classifier-free",
            condition=cond,
            unconditional_condition=uncond,
            guidance_scale=self.Config["w"] + 1,
        )
        dpm_solver = DPM_Solver(model_fn, self.noise_schedule)
        sampledImgs = dpm_solver.sample(
            noisyImage,
            steps=self.Config["dpm_solver_T"],
            order=self.Config["dpm_solver_order"],
            method=self.Config["dpm_solver_method"]
        )
        sampledImgs = sampledImgs * 0.5 + 0.5
        imgs = []
        # Save Images
        for b in range(self.Config["batch_size"]):
            box = lesion_box[b, 0, :4]
            save_img = os.path.join(
                self.Config["sampled_dir"],
                f"{cls_name}_{dev_name}" + \
                f"_x1_{float(box[0]):.4f}_y1_{float(box[1]):.4f}_x2_{float(box[2]):.4f}_y2_{float(box[3]):.4f}.png")
            save_image(sampledImgs[[b]], save_img, nrow=1)
            print(f"Save image to {save_img}")
            imgs.append(save_img)
        img = np.array(Image.open(imgs[0]))
        H, W = img.shape[:2]
        x1, y1, x2, y2 = float(box[0]), float(box[1]), float(box[2]), float(box[3])
        x1, y1, x2, y2 = int(W * x1), int(H * y1), int(W * x2), int(H * y2)
        img = np.concatenate([img, img], axis=1)
        img[y1:y1+1, x1:x2] = [255,0,0]
        img[y2-1:y2, x1:x2] = [255,0,0]
        img[y1:y2, x1:x1+1] = [255,0,0]
        img[y1:y2, x2-1:x2] = [255,0,0]
        return Image.fromarray(img)

### Build BUSGen model

In [ ]:
import Main
Main.load_env()  # Load .env if exists
Main.override_config_from_env

config = json.load(open("Configs/BUSGen_Sampling.json"))
config = Main.override_config_from_env(config)
config["batch_size"] = 1
model = BUSGen(config)

### Sampling with conditions

You need set the conditions for generating images by filling the **prompt window** following the requirements.

In [ ]:
def conditional_sampleing():
    cls_name = input("Please choose the pathology you want to generate: benign / malignant.")
    if cls_name.lower() not in ["benign", "malignant"]:
        raise ValueError(f"Pathology should be benign / malignant, but receive {cls_name}.")

    dev_name = input(f"Please choose the device name: {list(DEVICE_dict_all.keys())[1:]}.")
    if dev_name not in DEVICE_dict_all:
        raise ValueError(f"Device should be in {list(DEVICE_dict_all.keys())[1:]}, but receive {dev_name}.")
    
    lesion_box = input(
        f"Please choose the lesion location with x_top_left, y_top_left, x_bottum_right, y_bottum_right."
        "Values are in [0, 1]. For example, you can input 0.1 0.1 0.9 0.9 as coordinations."
    ).split()
    lesion_box = [float(x) for x in lesion_box]
    if not len(lesion_box) == 4:
        raise ValueError(f"Lesion location should have 4 coordinates but receive {len(lesion_box)} values.")
    if not all((0 <= x) and (x <= 1) for x in lesion_box):
        raise ValueError("lesion_box values should be in [0, 1].")
    if not lesion_box[0] < lesion_box[2] and lesion_box[1] < lesion_box[3]:
        raise ValueError("lesion_box should be in [x1, y1, x2, y2] format, where x1 < x2 and y1 < y2.")

    print(f"Generating image with a {cls_name} lesion, {dev_name} device and the lesion is located at {lesion_box}")
    img = model(cls_name, dev_name, lesion_box)
    return img

In [ ]:
conditional_sampleing()